# Diabetes Readmission Project
The dataset contains information about diabetic patients who were hospitalized over a 10-year period across 130 US hospitals, not necessarily for diabetes management, but flagged as diabetic during their stay. The goal is to identify a set of clinical and operational indicators that predict whether a patient will be readmitted in the future.

At first glance, this dataset needed to be handled carefully. Of the 101,766 entries, only roughly 71,000 represent unique patients, meaning duplicate encounter handling is required before any modeling can begin. Beyond that, missing values were encoded as '?' rather than left empty, causing the UCI data dictionary to report zero null values, a documentation failure caught only through direct inspection of the data itself.

Key clinical features including A1C, glucose levels, and weight carry extreme missingness rates, each requiring deliberate scoping decisions rather than standard imputation. Additionally, roughly 2% of patients had 5 or more admissions over the study period, with one extreme case reaching 40 encounters. This chronic readmission subgroup represents patients that standard discharge planning repeatedly failed to address.

### Note to the reader:

Key features include HbA1c result (a measure of average blood glucose over 3 months), primary and secondary diagnosis codes (ICD-9 format), medication records for 23 diabetes-related drugs, and prior utilization metrics including number of inpatient, outpatient, and emergency visits in the year before admission. The target variable is whether the patient was readmitted to the hospital after discharge.

# Preprocess Thoughts

The concern of hospital readmissions for patients with diabetes is the main concern for this project. Currently there are 3 result classes in our dataset ("NONE", ">30", "<30"). Since we are concerned with readmissions, we will separate these into a binary class, where NONE is a 0 and both >30 and <30 are consolidated into a single class 1. To further narrow down what we'll need and keep our data easier to manage, we'll be focusing on what the hospital can do while the patient is admitted to predict if they'll be readmitted.

Payer code was excluded from modeling to focus on factors within hospital control. This scoping decision means the model will not capture socioeconomic contributors to readmission, which are acknowledged as clinically significant but outside the scope of this analysis.

Due to the high levels of information infidelity, Weight and Medical Specialty columns will be getting dropped from analysis

Race, having a 2% missing information, will have a new category "Unknown" added to the column, so as not to introduce bias by imputing data.

Max glucose serum was excluded due to 94% missingness and lack of contextual metadata needed for clinical interpretation

Similar to Race, A1C tests will be given a new category of "None" to distinguish when no test was ordered or recorded

In [16]:
import pandas as pd
import sqlite3

df = pd.read_csv('../data/diabetic_data.csv')
conn =sqlite3.connect('../diabetes-readmission.db')
df.to_sql('encounters', conn, if_exists="replace", index=False)
conn.close()


In [17]:
print(f"Dataset shape: {df.shape}")
print(f"Total encounters: {len(df):,}")
print(f"Unique patients: {df['patient_nbr'].nunique():,}")
print(f"Duplicate encounters: {len(df) - df['patient_nbr'].nunique():,}")

Dataset shape: (101766, 50)
Total encounters: 101,766
Unique patients: 71,518
Duplicate encounters: 30,248


In [18]:
print(df['readmitted'].value_counts())
print(df['readmitted'].value_counts(normalize=True))

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64
readmitted
NO     0.539119
>30    0.349282
<30    0.111599
Name: proportion, dtype: float64


In [19]:
conn = sqlite3.connect('../diabetes-readmission.db')

null_check = pd.read_sql("""
    SELECT 
        COUNT(*),
        ROUND( 100 * SUM(CASE WHEN weight IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS weight_nulls_perc,
        ROUND( 100 * SUM(CASE WHEN race IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS race_nulls_perc,
        ROUND( 100 * SUM(CASE WHEN payer_code IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS payer_code_nulls_perc,
        ROUND( 100 * SUM(CASE WHEN medical_specialty IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS medical_specialty_nulls_perc,
        ROUND( 100 * SUM(CASE WHEN A1Cresult IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS A1C_null_perc,
        ROUND( 100 * SUM(CASE WHEN max_glu_serum IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS serum_null_perc
    FROM encounters                                  
""", conn)

print(null_check)
conn.close()

   COUNT(*)  weight_nulls_perc  race_nulls_perc  payer_code_nulls_perc  \
0    101766                0.0              0.0                    0.0   

   medical_specialty_nulls_perc  A1C_null_perc  serum_null_perc  
0                           0.0           83.0             94.0  


In [20]:
conn = sqlite3.connect('../diabetes-readmission.db')

encounter_dist = pd.read_sql("""
    SELECT 
        encounter_count,
        COUNT(*) as patient_count
    FROM (
        SELECT patient_nbr, COUNT(*) as encounter_count
        FROM encounters
        GROUP BY patient_nbr
    )
    GROUP BY encounter_count
    ORDER BY encounter_count
""", conn)

conn.close()
print(encounter_dist)

    encounter_count  patient_count
0                 1          54745
1                 2          10434
2                 3           3328
3                 4           1421
4                 5            717
5                 6            346
6                 7            207
7                 8            111
8                 9             70
9                10             42
10               11             20
11               12             19
12               13             14
13               14              5
14               15              9
15               16              4
16               17              3
17               18              6
18               19              3
19               20              6
20               21              1
21               22              2
22               23              3
23               28              1
24               40              1


## Next Steps

With a clear understanding of the dataset's structure and limitations, the next notebook will investigate the clinical indicators most likely to signal readmission risk. Starting with A1C levels, age, and primary and secondary diagnoses, the EDA will examine statistical distributions per class, compare the spread of readmitted vs non-readmitted patients across key features, and identify early signals worth carrying into feature engineering and modeling.